# PPG vs PPO parity (full ant, variable sampled morphs)

Report for `experiments/ppg_parity.py` (3 seeds each). Run that first to produce
`data/ppg_parity/{curves.npz,inference.npz}`. Comparison-only — no pass/fail gate.

1. performance over time (`morph_reward/mean`)
2. actor / critic / aux losses over time
3. phase timing (PPO rollout/update vs PPG rollout/policy/value/aux)
4. final-inference return on all 131 stable bodies (overall + per leg-count)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('../data/ppg_parity')
D = np.load(DATA / 'curves.npz')
I = np.load(DATA / 'inference.npz')
seeds = list(D['seeds'])
algos = [str(a) for a in D['algos']]
COLORS = {'ppo': 'C0', 'ppg': 'C1'}
print('seeds', seeds, '| algos', algos)

def series(algo, tag):
    """(steps, M[n_seeds, T]) for a tag across seeds; None if absent. Truncates to min T."""
    tg = tag.replace('/', '_')
    cols, steps = [], None
    for s in seeds:
        k = f'{algo}_s{s}__{tg}'
        if k + '__val' in D.files:
            cols.append(D[k + '__val'])
            st = D[k + '__step']
            steps = st if steps is None or len(st) < len(steps) else steps
    if not cols:
        return None, None
    T = min(len(c) for c in cols)
    return steps[:T], np.stack([c[:T] for c in cols])

def band(ax, algo, tag, label=None, warmup=1):
    x, M = series(algo, tag)
    if x is None:
        return False
    x, M = x[warmup:], M[:, warmup:]
    m, sd = M.mean(0), M.std(0)
    ax.plot(x, m, color=COLORS[algo], label=label or algo.upper())
    ax.fill_between(x, m - sd, m + sd, color=COLORS[algo], alpha=0.2)
    return True

## 1. Performance over time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for a in algos:
    band(ax, a, 'morph_reward/mean')
ax.set_xlabel('frame'); ax.set_ylabel('morph_reward/mean (shaped)')
ax.set_title('Performance over time (3-seed mean ± std)')
ax.legend(); plt.tight_layout()

## 2. Losses over time

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for a in algos:
    band(axs[0], a, 'losses/a_loss'); band(axs[1], a, 'losses/c_loss')
axs[0].set_title('actor loss'); axs[1].set_title('critic loss')
for ax in axs[:2]:
    ax.set_xlabel('frame'); ax.legend()
# PPG-only aux losses
for tag, lab in [('losses/aux_value', 'aux_value (head)'),
                 ('losses/aux_value_net', 'aux_value (net)'),
                 ('losses/aux_clone_kl', 'clone KL')]:
    x, M = series('ppg', tag)
    if x is not None:
        axs[2].plot(x, M.mean(0), label=lab)
axs[2].set_title('PPG aux losses'); axs[2].set_xlabel('frame'); axs[2].legend()
plt.tight_layout()

## 3. Phase timing

Steady-state per-epoch seconds (median over epochs, mean over seeds). `t_aux` is
logged only on aux-phase epochs (every `n_pi`); the amortized bar divides it by `n_pi`.

In [ ]:
N_PI = 32  # amortization window for t_aux

def med(algo, tag):
    x, M = series(algo, tag)
    return float(np.median(M[:, 1:])) if x is not None else 0.0  # drop epoch0 warmup

ppo_t = {'rollout': med('ppo', 'perf/t_rollout'), 'update': med('ppo', 'perf/t_update')}
ppg_t = {'rollout': med('ppg', 'perf/t_rollout'), 'policy': med('ppg', 'perf/t_policy'),
         'value': med('ppg', 'perf/t_value'), 'aux/n_pi': med('ppg', 'perf/t_aux') / N_PI}

fig, ax = plt.subplots(figsize=(9, 5))
def stack(x, parts, color0):
    bottom = 0
    for i, (k, v) in enumerate(parts.items()):
        ax.bar(x, v, 0.6, bottom=bottom, label=k, color=f'C{color0 + i}')
        if v > 0:
            ax.text(x, bottom + v / 2, f'{k}\n{v:.3f}', ha='center', va='center', fontsize=8)
        bottom += v
    return bottom
tot_ppo = stack(0, ppo_t, 0); tot_ppg = stack(1, ppg_t, 2)
ax.set_xticks([0, 1]); ax.set_xticklabels([f'PPO\n{tot_ppo:.3f}s', f'PPG\n{tot_ppg:.3f}s'])
ax.set_ylabel('seconds / epoch (amortized)')
ax.set_title('Per-epoch phase timing (steady-state)')
plt.tight_layout()
print('PPO', ppo_t, '\nPPG', ppg_t)

## 4. Final-inference return (all 131 stable bodies, deterministic mu)

In [ ]:
lc = I['bodies_legcount']
ret = {a: I[f'ret_{a}'] for a in algos if f'ret_{a}' in I.files}  # (n_seeds, n_bodies)

# overall: per-seed mean over bodies -> mean +/- std over seeds
print('Final-inference return (mean over bodies):')
for a, R in ret.items():
    per_seed = R.mean(1)
    print(f'  {a.upper():4s}  {per_seed.mean():8.1f} +/- {per_seed.std():6.1f}   per-seed {np.round(per_seed,1)}')

counts = sorted(set(int(c) for c in lc))
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(counts)); w = 0.8 / max(len(ret), 1)
for j, (a, R) in enumerate(ret.items()):
    means = [R[:, lc == c].mean() for c in counts]
    errs = [R[:, lc == c].mean(1).std() for c in counts]
    ax.bar(x + (j - (len(ret) - 1) / 2) * w, means, w, yerr=errs, label=a.upper(), color=COLORS[a])
ax.set_xticks(x); ax.set_xticklabels(counts)
ax.set_xlabel('leg count'); ax.set_ylabel('mean return')
ax.set_title('Final-inference return per leg-count (seed mean ± std)')
ax.legend(); plt.tight_layout()